# Recommendation 05 — Confronto tramite la pipeline LLM+MCP **lineare**

**Domanda:** a parità di 1, 3, 5 e 10 rating rivelati, quali ranking ottiene la *pipeline reale* (Planner LLM → MCP `search_resources` → retrieval GTE → Answer LLM) rispetto a Popularity, MF e GTE diretto?

Non è il notebook 04 (LLM che genera direttamente una query fuori dalla pipeline): qui viene chiamata la factory **`build_resource_assistant_workflow`** già presente nel progetto, con configurazione MovieLens v2, GTE fast **senza reranker**. Le richieste sono **deterministiche** e contengono soltanto **titoli e rating**.

**Importante sul ranking:** `state.final_answer.answer` è testo e `state.final_answer.recommendations` può avere meno di 10 elementi. Il ranking quantitativo è derivato direttamente da `state.search_response.results` (ordine MCP) senza leggere/interpretare il testo. L'Answer LLM è comunque eseguito e salvato per diagnosi. Questo misura il **retrieval guidato dal Planner**, non la capacità dell'Answer agent di scegliere la top10: il secondo è un task distinto.

Per ottenere 10 elementi richiediamo fino a 30 candidati MCP e conserviamo i primi 10 film validi **non ancora rivelati**. Se ne restano meno, registriamo l'insufficienza senza riempire con Popularity. NDCG/Recall/HR considerano mancanti le posizioni non restituite.

**Valutazione** A = 20 target fissi; B = tutti i rating nascosti. Nessun LLM simulatore per lo zero-shot: `k=0` riproduce **Popularity**. Il dialogo di cold-start sarà una fase distinta.

**Confine anti-leakage:** le celle che generano ranking NON importano `private_evaluation/`. Il ground truth è letto soltanto nella cella finale di valutazione, dopo il salvataggio delle predizioni. Le interazioni degli utenti di test non vengono usate per allenare il retriever.

## 0. Configurazione e ambiente

La modalità predefinita esegue **solo controlli offline**. Dopo aver verificato le query, imposta `RUN_LIVE=True` per avviare il pilota validation. L'archivio più recente che abbiamo potuto verificare documenta la factory lineare; se i contratti della `src/` locale sono cambiati, aggiorna l'adapter specifico, non i componenti MF/GTE.

In [2]:
from pathlib import Path
import os, sys, json, hashlib, time, asyncio
import pandas as pd
import numpy as np
from IPython.display import display

PROJECT_ROOT_OVERRIDE = None   # o Path('/path/to/repository')
RUN_LIVE = True               # PRIMA esegui preview; poi True con API e Qdrant pronti
RUN_MODE = 'pilot_validation'  # 'pilot_validation', 'full_validation', 'test'
ALLOW_TEST = False             # per test finale: True + RUN_MODE='test' solo dopo blocco metodologico
PILOT_USERS = 6                # utenti distinti, una replica, fino a 24 esecuzioni reali
PILOT_REPLICA = 0
K_LEVELS = (0,1,3,5,10)
TEMPLATE_ID = 'title_rating_v1'  # estendibile nel file recommendation_05_pipeline_helpers.py
SEED = 20261021
TARGET_K = 10
RETRIEVAL_LIMIT = 30          # più alto di 10 per scartare film già visti
MAX_NEW_CALLS = 24            # numero massimo di invocazioni COMPLETE del workflow (planner+answer)
STOP_ON_ERROR = True

# Factory e profili esattamente come nel notebook 11 / orchestrazione lineare.
DOMAIN_ID = 'movielens_v2'
RESOURCE_RAG_PROFILE = 'movielens_gte_fast_v2'
WORKFLOW_ID = 'grounded_resource_v1'
PLANNER_LLM_PROFILE = 'openai_default' #'gemini_free'
ANSWER_LLM_PROFILE = 'openai_default' #'gemini_free'
QUERY_REWRITE_STRATEGY = 'translate_to_catalog_language'

PROTOCOL_ID = 'rs_coldstart_v1'
MODEL_ID = 'llm_pipeline_linear_retrieval'


def find_root():
    if PROJECT_ROOT_OVERRIDE is not None:
        return Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    here=Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d/'src/movie_rag').is_dir() and (d/'datasets/processed/recommendation_protocol'/PROTOCOL_ID/'protocol_manifest.json').exists():
            return d
    raise FileNotFoundError('Root progetto non individuata; imposta PROJECT_ROOT_OVERRIDE')
ROOT=find_root()
if RUN_MODE not in {'pilot_validation','full_validation','test'}: raise ValueError('RUN_MODE non valido')
if RUN_MODE=='test' and not ALLOW_TEST: raise RuntimeError('Il test è bloccato; fissare prima le scelte in validation')
SRC=ROOT/'src'
if str(SRC) not in sys.path: sys.path.insert(0,str(SRC))
HERE=Path.cwd().resolve()
if str(HERE) not in sys.path: sys.path.insert(0,str(HERE))
from recommendation_05_pipeline_helpers import make_query, input_digest, extract_rankings, score_rankings, macro_by_user, TEMPLATES
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT/'.env',override=False)
except ImportError:
    print('python-dotenv non trovato: credenziali già nell’ambiente necessarie per RUN_LIVE')
BASE=ROOT/'datasets/processed/recommendation_protocol'/PROTOCOL_ID
IN=BASE/'inputs'; PRIV=BASE/'private_evaluation'; PRED=BASE/'predictions'
OUT=BASE/'evaluations'/'llm_pipeline_linear_v1'/RUN_MODE
OUT.mkdir(parents=True,exist_ok=True)
assert TEMPLATE_ID in TEMPLATES
print('ROOT:',ROOT,' | RUN:',RUN_MODE,' | OUTPUT:',OUT,' | TEMPLATE:',TEMPLATE_ID)

ROOT: /Users/danieleraveglia/Documents/uniMi/TESI/progetto  | RUN: pilot_validation  | OUTPUT: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/datasets/processed/recommendation_protocol/rs_coldstart_v1/evaluations/llm_pipeline_linear_v1/pilot_validation  | TEMPLATE: title_rating_v1


## 1. Leggi solo dati pubblici al recommender e costruisci le query

La funzione del template è registrata in `TEMPLATES`, quindi per arricchirla in futuro basterà registrare una nuova variante **versionata**. Non cambiare un template già utilizzato e non includere target nascosti.

In [3]:
manifest=json.loads((BASE/'protocol_manifest.json').read_text(encoding='utf-8'))
assert manifest['protocol_id']==PROTOCOL_ID
scenarios=pd.read_csv(IN/'scenarios.csv')
levels=pd.read_csv(IN/'scenario_levels.csv')
visible=pd.read_csv(IN/'visible_interactions.csv')
movies=pd.read_csv(ROOT/'datasets/raw/ml-latest-small/movies.csv')
catalog=pd.read_csv(IN/'catalog_movie_ids.csv')
assert {'scenario_id','userId','split','rep'}.issubset(scenarios.columns)
assert {'scenario_id','k','movieId','rating'}.issubset(visible.columns)
assert {'scenario_id','k'}.issubset(levels.columns)
assert {'movieId','title'}.issubset(movies.columns)
for f in [visible,movies,catalog]:f['movieId']=f['movieId'].astype(int)
for f in [visible,levels]:f['k']=f['k'].astype(int)
all_ids=set(catalog.movieId.astype(int))
if catalog.movieId.duplicated().any() or movies.movieId.duplicated().any():raise ValueError('movieId duplicati')
titles=movies.set_index('movieId').title.to_dict()
if not all_ids.issubset(set(titles)):raise ValueError('Titoli MovieLens mancanti')
observed={(str(s),int(k)):g[['movieId','rating']].copy() for (s,k),g in visible.groupby(['scenario_id','k'])}
requested_split='test' if RUN_MODE=='test' else 'validation'
subset=scenarios[scenarios.split==requested_split].copy()
if RUN_MODE=='pilot_validation':
    subset=subset[subset.rep==PILOT_REPLICA]
    users=np.sort(subset.userId.unique())
    users=np.random.default_rng(SEED).choice(users,size=min(PILOT_USERS,len(users)),replace=False)
    subset=subset[subset.userId.isin(users)]
selected=levels[levels.scenario_id.isin(subset.scenario_id)&levels.k.isin(K_LEVELS)].sort_values(['scenario_id','k']).copy()
assert len(selected)==len(subset)*len(K_LEVELS)
query_records=[]
for rec in selected.itertuples(index=False):
    sid=str(rec.scenario_id);k=int(rec.k)
    obs=observed.get((sid,k),pd.DataFrame(columns=['movieId','rating']))
    assert len(obs)==k and obs.movieId.is_unique
    assert set(obs.movieId).issubset(all_ids)
    q=make_query(obs,titles,TEMPLATE_ID)
    query_records.append({'scenario_id':sid,'k':k,'query':q,'input_hash':input_digest(obs,titles,TEMPLATE_ID),
                          'revealed_ids':list(map(int,obs.movieId)), 'userId':int(subset.loc[subset.scenario_id==sid,'userId'].iloc[0])})
queries=pd.DataFrame(query_records)
print('Utenti:',subset.userId.nunique(),'| scenari:',len(subset),'| livelli:',len(selected),
      '| invocazioni pipeline k>0:',int((selected.k>0).sum()))
display(queries[['scenario_id','k','query']].head(5))

Utenti: 6 | scenari: 6 | livelli: 30 | invocazioni pipeline k>0: 24


,scenario_id,k,query
0,validation_u100_r0,0,Vorrei dei consigli su film da vedere. Non ho ...
1,validation_u100_r0,1,Consigliami dei film che potrebbero piacermi s...
2,validation_u100_r0,3,Consigliami dei film che potrebbero piacermi s...
3,validation_u100_r0,5,Consigliami dei film che potrebbero piacermi s...
4,validation_u100_r0,10,Consigliami dei film che potrebbero piacermi s...


In [7]:
queries['query'][4]

"Consigliami dei film che potrebbero piacermi sulla base delle valutazioni che ho fornito.\nLe stelle indicano quanto ho apprezzato ciascun film (scala 0,5-5):\n- [17] Sense and Sensibility (1995): 4.5/5 stelle\n- [296] Pulp Fiction (1994): 3.5/5 stelle\n- [900] American in Paris, An (1951): 4.0/5 stelle\n- [1220] Blues Brothers, The (1980): 3.0/5 stelle\n- [1246] Dead Poets Society (1989): 4.5/5 stelle\n- [2396] Shakespeare in Love (1998): 4.5/5 stelle\n- [2710] Blair Witch Project, The (1999): 2.0/5 stelle\n- [4621] Look Who's Talking (1989): 4.0/5 stelle\n- [6377] Finding Nemo (2003): 4.0/5 stelle\n- [8529] Terminal, The (2004): 4.0/5 stelle\nConsidera sia le valutazioni alte sia quelle basse.\nSuggerisci film diversi da quelli già indicati."

## 2. Configurazione della pipeline **reale** (senza eseguirla)

Si riutilizzano le factory già disponibili e i profili v2. `fast` indica **reranker OFF**. L'Answer agent viene eseguito normalmente, ma NON determina l'ordine del ranking di benchmark. Nessuna modifica al codice di `src/movie_rag`.

In [5]:
LLM_CONFIG=ROOT/'configs/llm_profiles_v1.json'
DOMAIN_CONFIG=ROOT/'configs/resource_domains_v1.json'
MCP_CONFIG=ROOT/'configs/resource_mcp_profiles_v1.json'
for f in (LLM_CONFIG,DOMAIN_CONFIG,MCP_CONFIG):
    if not f.is_file():raise FileNotFoundError(f)

def hash_file(p):
    h=hashlib.sha256();h.update(p.read_bytes());return h.hexdigest()

# Include modifiche locali del package nel fingerprint per non riutilizzare cache stale.
source_parts=['resource_llm_workflow.py','resource_llm_agents.py','resource_mcp_profiles.py','resource_mcp_gateway.py']
source_hashes={part:hash_file(ROOT/'src/movie_rag'/part) if (ROOT/'src/movie_rag'/part).is_file() else 'missing'
               for part in source_parts}
run_params={'protocol':PROTOCOL_ID,'mode':'linear','template_id':TEMPLATE_ID,
            'domain_id':DOMAIN_ID,'mcp_profile':RESOURCE_RAG_PROFILE,
            'workflow_id':WORKFLOW_ID,'planner_llm_profile':PLANNER_LLM_PROFILE,
            'answer_llm_profile':ANSWER_LLM_PROFILE,'rewrite_strategy':QUERY_REWRITE_STRATEGY,
            'requested_k':TARGET_K,'retrieval_limit':RETRIEVAL_LIMIT,
            'configs':{str(p.relative_to(ROOT)):hash_file(p) for p in (LLM_CONFIG,DOMAIN_CONFIG,MCP_CONFIG)},
            'sources':source_hashes, 'helper_hash':hash_file(Path(sys.modules['recommendation_05_pipeline_helpers'].__file__).resolve())}
CONFIG_HASH=hashlib.sha256(json.dumps(run_params,sort_keys=True).encode()).hexdigest()
print('Fingerprint configurazione:',CONFIG_HASH[:16])

Fingerprint configurazione: eb3db18cd2c94085


## 3. Runner del workflow lineare e trasformazione in ranking

Lo stesso server MCP è mantenuto aperto durante il pilota; il workflow è quello della factory esistente. `search_response.results` contiene hit con `resource_id`, `rank`, `score`. Dopo la risposta estraiamo fino a dieci film validi; conserviamo anche l'output dell'Answer agent, ma senza tentare di interpretarne il testo.

`RUN_LIVE=False` non chiama LLM, Qdrant né il modello GTE: puoi verificare template e configurazione prima di generare costi.

In [6]:
if RUN_LIVE:
    from mcp import Client
    from movie_rag.resource_domain import load_resource_domain_catalog
    from movie_rag.resource_mcp_profiles import ResourceSearchServiceRegistry, load_resource_mcp_profiles
    from movie_rag.resource_adapters import movie_filters_to_resource_filters
    from movie_rag.resource_mcp import create_resource_mcp_server
    from movie_rag.resource_mcp_gateway import MCPResourceToolGateway
    from movie_rag.resource_llm_workflow import build_resource_assistant_workflow
    from movie_rag import GeminiProvider, OpenAIProvider, load_llm_config

    os.environ['RESOURCE_RAG_PROFILE']=RESOURCE_RAG_PROFILE
    os.environ.setdefault('QDRANT_URL','http://localhost:6333')
    domain=load_resource_domain_catalog(DOMAIN_CONFIG).get(DOMAIN_ID)
    profile=load_resource_mcp_profiles(MCP_CONFIG).get(RESOURCE_RAG_PROFILE)
    assert domain.domain_id==profile.domain_id
    if profile.reranker is not None:
        raise RuntimeError('Benchmark lineare richiede FAST senza reranker. Controlla profilo MCP.')
    registry=ResourceSearchServiceRegistry.from_environment(MCP_CONFIG,profile_id=RESOURCE_RAG_PROFILE)
    assert registry.service.reranker is None, 'Reranker caricato in modalità FAST: interrompere'
    mcp_server=create_resource_mcp_server(
        registry.service,name='Evaluation 05 linear retrieval',
        filter_adapter=movie_filters_to_resource_filters,
        default_profile_id=registry.profile.search_profile_id,
        expose_profile_to_client=False,exact=registry.profile.exact)
    llm_config=load_llm_config(LLM_CONFIG)
    planner_provider=OpenAIProvider(llm_config.get_llm_profile(PLANNER_LLM_PROFILE))
    answer_provider=OpenAIProvider(llm_config.get_llm_profile(ANSWER_LLM_PROFILE))
    print('Pipeline lineare pronta:',RESOURCE_RAG_PROFILE)
else:
    print('Preview offline: nessuna chiamata LLM/MCP/Qdrant')

async def run_single_pipeline(workflow,query_text):
    return await workflow.run(query_text)

Pipeline lineare pronta: movielens_gte_fast_v2


[10/30/26 12:41:23] INFO     HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"              ]8;id=1720556;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=1720557;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\

## 4. Esecuzione pilota, checkpoint e protezione dei costi

La cache JSONL usa hash di template, rating visibili, configurazione e sorgenti del workflow. I casi già completati vengono riutilizzati. Un errore interrompe di default l'esperimento; non viene sostituito silenziosamente da un ranking di Popularity. Il fallback a `k=0` è invece **definito dal protocollo** e usa Popularity già calcolata nel notebook 02.

In [7]:
BASELINE_FILE=PRED/('rankings_baselines_test.csv' if requested_split=='test' else 'rankings_baselines_validation.csv')
if not BASELINE_FILE.is_file():raise FileNotFoundError(BASELINE_FILE)
baselines=pd.read_csv(BASELINE_FILE)
base_model='gte_direct' if requested_split=='test' else None
if requested_split=='validation':
    conf=json.loads((BASE/'evaluations/baseline_mf_gte/model_config.json').read_text(encoding='utf-8'))
    base_model=f'gte_negalpha_{float(conf["gte_neg_alpha_selected"]):g}'
if base_model not in set(baselines.model):raise ValueError(f'Non trovata baseline {base_model}')
CACHE=OUT/'workflow_runs.jsonl'
cache={}
if CACHE.exists():
    for line in CACHE.read_text(encoding='utf-8').splitlines():
        if not line.strip():continue
        obj=json.loads(line)
        cache[(str(obj['scenario_id']),int(obj['k']),obj['input_hash'],obj['config_hash'])]=obj
print('Run in cache:',len(cache))

needed=[]
for q in queries.itertuples(index=False):
    key=(str(q.scenario_id),int(q.k),q.input_hash,CONFIG_HASH)
    if key not in cache and q.k>0:needed.append(key)
if RUN_LIVE and len(needed)>MAX_NEW_CALLS:
    raise RuntimeError(f'Servono {len(needed)} chiamate workflow > MAX_NEW_CALLS={MAX_NEW_CALLS}. Riduci batch o alza limite.')

async def execute_batch():
    if not RUN_LIVE:
        print('RUN_LIVE=False: genera soltanto query di anteprima e verifica impostazioni')
        return []
    saved=[]
    async with Client(mcp_server,raise_exceptions=True) as client:
        gateway=MCPResourceToolGateway(client)
        workflow=build_resource_assistant_workflow(
            domain=domain,planner_provider=planner_provider,answer_provider=answer_provider,
            gateway=gateway,workflow_id=WORKFLOW_ID,
            result_limit=RETRIEVAL_LIMIT,query_rewrite_strategy=QUERY_REWRITE_STRATEGY)
        for q in queries.itertuples(index=False):
            sid,k=str(q.scenario_id),int(q.k)
            key=(sid,k,q.input_hash,CONFIG_HASH)
            if key in cache:
                saved.append(cache[key]);continue
            if k==0:
                # Identico fallback al notebook 02, senza uso LLM.
                sub=baselines[(baselines.scenario_id==sid)&(baselines.k==0)&(baselines.model=='popularity')].sort_values('rank')
                if len(sub)!=TARGET_K:raise ValueError(f'Popularity k0 non completa: {sid}')
                retrieved=[{'movieId':int(r.movieId),'rank':int(r.rank),'score':float(r.score),'title':''} for r in sub.itertuples(index=False)]
                result={'retrieval':retrieved,'answer_ids':[],'answer_unmapped':[],
                        'raw_retrieved':0,'eligible_retrieved':TARGET_K,'shown_excluded':[],
                        'returned':TARGET_K,'shortfall':0,'state_status':'K0_POPULARITY',
                        'mcp_status':'not_called','mcp_query':None,
                        'final_answer_text':None,'response_missing':False}
                latency=0.0
            else:
                tic=time.perf_counter()
                try:
                    state=await run_single_pipeline(workflow,q.query)
                    result=extract_rankings(state,catalog=all_ids,shown=set(q.revealed_ids),target_k=TARGET_K)
                except Exception as exc:
                    print(f'ERRORE {sid} k={k}: {type(exc).__name__}: {exc}')
                    if STOP_ON_ERROR: raise
                    continue
                latency=time.perf_counter()-tic
            row={'scenario_id':sid,'k':k,'input_hash':q.input_hash,'config_hash':CONFIG_HASH,
                 'template_id':TEMPLATE_ID,'query':q.query,'latency_seconds':latency,**result}
            with CACHE.open('a',encoding='utf-8') as f:
                f.write(json.dumps(row,ensure_ascii=False,default=str)+chr(10))
            saved.append(row);cache[key]=row
            if len(saved)%10==0:print('Salvati:',len(saved),'/',len(queries))
    return saved

runs=await execute_batch()
print('Run caricati/eseguiti:',len(runs),'| attesi:',len(queries))
if runs:
    overview=pd.DataFrame([{k:v for k,v in row.items() if k not in {'retrieval','final_answer_text'}} for row in runs])
    display(overview[['scenario_id','k','state_status','mcp_status','raw_retrieved','returned','shortfall','latency_seconds']].head(20))

Run in cache: 32


[10/30/26 12:41:27] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 400   ]8;id=1720563;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=1720564;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\
                             Bad Request"                                                                          

[10/30/26 12:41:35] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=1720569;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=1720570;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

[10/30/26 12:41:40] INFO     No device provided, using mps                                             ]8;id=1720577;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/sentence_transformers/base/model.py\model.py]8;;\:]8;id=1720578;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/sentence_transformers/base/model.py#190\190]8;;\

[10/30/26 12:41:42] INFO     Loading SentenceTransformer model from                                   ]8;id=1720584;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/sentence_transformers/base/model.py\model.py]8;;\:]8;id=1720585;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/sentence_transformers/base/model.py#1022\1022]8;;\
                             Alibaba-NLP/gte-multilingual-base.                                                    

Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


... (118 outputs left out)

[10/30/26 12:49:53] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 400   ]8;id=1721286;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=1721287;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\
                             Bad Request"                                                                          

[10/30/26 12:50:07] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=1721292;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=1721293;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

Salvati: 30 / 30
Run caricati/eseguiti: 30 | attesi: 30


,scenario_id,k,state_status,mcp_status,raw_retrieved,returned,shortfall,latency_seconds
0,validation_u100_r0,0,K0_POPULARITY,not_called,0,10,0,0.000000
1,validation_u100_r0,1,completed,ok,10,9,1,40.886430
2,validation_u100_r0,3,completed,ok,10,8,2,21.035419
3,validation_u100_r0,5,completed,ok,10,9,1,21.186111
4,validation_u100_r0,10,completed,ok,10,10,0,23.200617
5,validation_u10_r0,0,K0_POPULARITY,not_called,0,10,0,0.000000
6,validation_u10_r0,1,completed,ok,10,9,1,20.945150
7,validation_u10_r0,3,completed,ok,10,8,2,20.629129
8,validation_u10_r0,5,completed,ok,10,8,2,20.002223
9,validation_u10_r0,10,completed,ok,10,6,4,20.636630


In [8]:
display(overview[['scenario_id','k','state_status','mcp_status','raw_retrieved','returned','shortfall','latency_seconds']])

,scenario_id,k,state_status,mcp_status,raw_retrieved,returned,shortfall,latency_seconds
0,validation_u100_r0,0,K0_POPULARITY,not_called,0,10,0,0.000000
1,validation_u100_r0,1,completed,ok,10,9,1,40.886430
2,validation_u100_r0,3,completed,ok,10,8,2,21.035419
3,validation_u100_r0,5,completed,ok,10,9,1,21.186111
4,validation_u100_r0,10,completed,ok,10,10,0,23.200617
5,validation_u10_r0,0,K0_POPULARITY,not_called,0,10,0,0.000000
6,validation_u10_r0,1,completed,ok,10,9,1,20.945150
7,validation_u10_r0,3,completed,ok,10,8,2,20.629129
8,validation_u10_r0,5,completed,ok,10,8,2,20.002223
9,validation_u10_r0,10,completed,ok,10,6,4,20.636630


In [9]:
# n-esima riga, con n indicizzato da zero
overview.iloc[1]["query"]

'Consigliami dei film che potrebbero piacermi sulla base delle valutazioni che ho fornito.\nLe stelle indicano quanto ho apprezzato ciascun film (scala 0,5-5):\n- [17] Sense and Sensibility (1995): 4.5/5 stelle\nConsidera sia le valutazioni alte sia quelle basse.\nSuggerisci film diversi da quelli già indicati.'

In [10]:
runs

[{'scenario_id': 'validation_u100_r0',
  'k': 0,
  'input_hash': 'b8dc0b6c14e844a1e1082058d87eefdffd2e7fc2ae999f782c7dc17d817124c9',
  'config_hash': 'eb3db18cd2c940851bd05f49eba311f6891fbb655d21c633e042e0406d71a858',
  'template_id': 'title_rating_v1',
  'query': 'Vorrei dei consigli su film da vedere. Non ho ancora indicato preferenze.',
  'latency_seconds': 0.0,
  'retrieval': [{'movieId': 318,
    'rank': 1,
    'score': 4.375867809882516,
    'title': ''},
   {'movieId': 858, 'rank': 2, 'score': 4.246966521578114, 'title': ''},
   {'movieId': 58559, 'rank': 3, 'score': 4.235804990621686, 'title': ''},
   {'movieId': 2959, 'rank': 4, 'score': 4.218668274313955, 'title': ''},
   {'movieId': 1136, 'rank': 5, 'score': 4.183202572235661, 'title': ''},
   {'movieId': 1197, 'rank': 6, 'score': 4.182799257465686, 'title': ''},
   {'movieId': 1213, 'rank': 7, 'score': 4.178124423202429, 'title': ''},
   {'movieId': 2329, 'rank': 8, 'score': 4.178124423202429, 'title': ''},
   {'movieId': 2

## 5. Formato ranking ed evidenze (nessun ground truth)

Sono registrati **due canali distinti**: ranking MCP e ID citati nella risposta LLM. Le metriche principali usano solo il ranking MCP. I candidati già mostrati all'utente vengono filtrati **dopo il retrieval** e il relativo conteggio viene registrato.

In [12]:
rank_rows=[]; answer_rows=[]; missing=[]
for r in runs:
    for hit in r['retrieval']:
        rank_rows.append({'scenario_id':r['scenario_id'],'k':int(r['k']),'model':MODEL_ID,
                          'movieId':int(hit['movieId']),'rank':int(hit['rank']),
                          'score':hit['score']})
    for j,mid in enumerate(r.get('answer_ids',[]),1):
        answer_rows.append({'scenario_id':r['scenario_id'],'k':int(r['k']),
                            'rank':j,'movieId':int(mid)})
    if r['shortfall']>0:missing.append((r['scenario_id'],r['k'],r['shortfall']))
rankings=pd.DataFrame(rank_rows,columns=['scenario_id','k','model','movieId','rank','score'])
answer_ranking=pd.DataFrame(answer_rows,columns=['scenario_id','k','rank','movieId'])
if runs:
    if len(runs)!=len(queries):
        raise RuntimeError('Esecuzione incompleta: non valutare finché tutti gli scenari sono salvati')
    assert not rankings[['scenario_id','k','movieId']].duplicated().any()
    assert rankings.movieId.isin(all_ids).all()
    assert (rankings.groupby(['scenario_id','k']).size()<=TARGET_K).all()
    for r in runs:
        shown=set(queries[(queries.scenario_id==r['scenario_id'])&(queries.k==r['k'])].iloc[0].revealed_ids)
        assert not shown.intersection({x['movieId'] for x in r['retrieval']})
    rankings.to_csv(OUT/'rankings_llm_pipeline_mcp.csv',index=False)
    answer_ranking.to_csv(OUT/'answer_recommendation_ids.csv',index=False)
    pd.DataFrame([{'scenario_id':r['scenario_id'],'k':r['k'],'query':r['query'],
                   'mcp_query':r.get('mcp_query'), 'state_status':r['state_status'],
                   'retrieval_count':r['returned'],'answer_count':len(r['answer_ids']),
                   'answer_text':r.get('final_answer_text'), 'duration_seconds':r['latency_seconds']}
                  for r in runs]).to_csv(OUT/'pipeline_traces.csv',index=False)
    print('Ranking MCP salvati:',len(rankings),'| liste incomplete:',len(missing),missing[:10])
else:
    print('Nessun ranking: impostare RUN_LIVE=True. Nessun accesso al ground truth effettuato.')

Ranking MCP salvati: 259 | liste incomplete: 23 [('validation_u100_r0', 1, 1), ('validation_u100_r0', 3, 2), ('validation_u100_r0', 5, 1), ('validation_u10_r0', 1, 1), ('validation_u10_r0', 3, 2), ('validation_u10_r0', 5, 2), ('validation_u10_r0', 10, 4), ('validation_u129_r0', 1, 1), ('validation_u129_r0', 3, 2), ('validation_u129_r0', 5, 2)]


## 6. Valutazione A e B — solo dopo aver salvato il ranking

**Non modificare i template o il Planner in funzione del risultato di test:** per ora esegui il pilota e, se necessario, aggiusta su validation. I film senza rating non sono necessariamente negativi. Anche un ranking sotto 10 elementi viene valutato con le posizioni mancanti a guadagno nullo e il deficit riportato.

In [13]:
if runs and len(runs)==len(queries):
    from recommendation_05_pipeline_helpers import score_rankings,macro_by_user
    fixed=pd.read_csv(PRIV/'private_fixed_gt.csv')
    extra=pd.read_csv(PRIV/'private_additional.csv')
    profile=pd.read_csv(PRIV/'private_profile.csv')
    # I baseline dello stesso subset: GTE diretto e Popularity e MF selezionato (se presente).
    allowed=['popularity',base_model]
    if requested_split=='test':
        allowed+=['mf_explicit_foldin']
    else:
        cf=json.loads((BASE/'evaluations/baseline_mf_gte/model_config.json').read_text(encoding='utf-8'))
        allowed+=[f'mf_foldreg_{float(cf["mf_fold_reg_selected"]):g}']
    comparison=baselines[baselines.scenario_id.isin(selected.scenario_id)&
                         baselines.k.isin(K_LEVELS)&baselines.model.isin(allowed)].copy()
    name_map={base_model:'gte_direct'}
    for m in allowed:
        if m.startswith('mf_foldreg_'):name_map[m]='mf_explicit_foldin'
    comparison['model']=comparison.model.replace(name_map)
    union=pd.concat([rankings,comparison],ignore_index=True)
    expected={str(x) for x in selected.scenario_id}
    models={MODEL_ID,'gte_direct','popularity','mf_explicit_foldin'}
    actual=union.groupby(['scenario_id','k']).model.apply(set)
    assert len(actual)==len(selected) and all(ms==models for ms in actual)
    expected=pd.DataFrame([{'scenario_id':str(r.scenario_id),'k':int(r.k),'model':m}
                           for r in selected.itertuples(index=False) for m in sorted(models)])
    metrics=score_rankings(union,fixed=fixed,extra=extra,private_profile=profile,top_k=TARGET_K,expected_keys=expected)
    owner=dict(zip(scenarios.scenario_id,scenarios.userId))
    per_user,macro=macro_by_user(metrics,owner)
    metrics.to_csv(OUT/'metrics_per_scenario_A_B.csv',index=False)
    per_user.to_csv(OUT/'metrics_per_user_A_B.csv',index=False)
    macro.to_csv(OUT/'macro_A_B.csv',index=False)
    report={'input_template':TEMPLATE_ID,'run_mode':RUN_MODE,'fingerprint':CONFIG_HASH,
            'model':MODEL_ID,'source':'MCP search results (not Answer LLM)',
            'answer_recommendations_are_secondary':True,
            'full_lists':sum(r['shortfall']==0 for r in runs),
            'short_lists':sum(r['shortfall']>0 for r in runs),
            'scenarios':len(runs),'catalog_size':len(all_ids),
            'policy':'No popularity padding; K0 uses baseline popularity; no hidden ratings in requests',
            'config':run_params}
    (OUT/'manifest_run.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
    display(macro[['model','scheme','k','ndcg@10','recall@10','hr@10','shortfall']].round(4))
    print('Salvati report:',OUT)
else:
    print('Valutazione non eseguita senza ranking completi. Nessun file privato letto.')

,model,scheme,k,ndcg@10,recall@10,hr@10,shortfall
0,gte_direct,A,0,0.0663,0.0537,0.3333,0.0000
1,gte_direct,B,0,0.1998,0.0431,1.0000,0.0000
2,llm_pipeline_linear_retrieval,A,0,0.0663,0.0537,0.3333,0.0000
3,llm_pipeline_linear_retrieval,B,0,0.1998,0.0431,1.0000,0.0000
4,mf_explicit_foldin,A,0,0.0663,0.0537,0.3333,0.0000
5,mf_explicit_foldin,B,0,0.1998,0.0431,1.0000,0.0000
6,popularity,A,0,0.0663,0.0537,0.3333,0.0000
7,popularity,B,0,0.1998,0.0431,1.0000,0.0000
8,gte_direct,A,1,0.0000,0.0000,0.0000,0.0000
9,gte_direct,B,1,0.0950,0.0127,0.5000,0.0000


Salvati report: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/datasets/processed/recommendation_protocol/rs_coldstart_v1/evaluations/llm_pipeline_linear_v1/pilot_validation


## 7. Interpretazione e prossimi passi

1. Controllare 5–10 richieste e confrontare **query inviata**, **query MCP riformulata**, **Top10 MCP**, **ID citati nell'Answer LLM** e **risposta testuale**. Se la risposta finale cita meno film, **non significa** che MCP ne abbia recuperati meno.
2. Verificare `shortfall`: se molti casi restituiscono meno di dieci film, controllare filtri generati dal Planner e numero di candidati disponibili, **senza inventare item**.
3. Per le metriche principali si usa MCP top10; per studiare quanto l'Answer seleziona tra candidati, serve una valutazione secondaria con le sue raccomandazioni, possibilmente più corte.
4. Per cambiare il template in modo riproducibile aggiungi una funzione al registry `TEMPLATES` con **un nuovo ID**, e confronta solo su validation. Il file `rankings_baselines_validation.csv` resta immutato.
5. Confronto finale: dopo aver fissato il template e la configurazione, esegui full_validation. I risultati precedenti sono già stati analizzati sul vecchio test, quindi un'ulteriore valutazione su quello stesso test è *esplorativa*; per affermazioni confermative servirà una partizione non consultata.
6. **Cold-start con interazione fra due LLM non è incluso**: sarà una nuova fase, dove il simulatore conosce soltanto il bucket privato P di 10 rating, senza accesso al ground truth A/B.